# J-Space Experiment — Phases 1–4 (Qwen3.5-9B instruct)

This is the full Qwen3.5-9B instruct Colab launcher. The configure cell defaults to `RUN_MODE = 'full'` and `configs/phase1_full_qwen.yaml`. Keep `smoke` only if you need the short EmailQA pipeline. After Drive is mounted, the next cell inventories the folders and files the full run needs. Each phase uses the same CLI as a local or SSH run. Results stay under one run root; the notebook stores no scientific state outside that directory.

## 1. Install the experiment and pinned benchmark checkouts

In [1]:
import subprocess
from pathlib import Path

RESEARCH_REPO = 'https://github.com/ethanncyb/jspace-research.git'
RESEARCH_REVISION = 'prompt-injection-experiment'
BIPIA_REVISION = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
AGENTDOJO_REVISION = '089ed468cf3ed0322acc66b0211f26d9d90dbf60'
INJECAGENT_REVISION = 'f19c9f2c79a41046eb13c03c51a24c567a8ffa07'


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def run_checked(command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        print('Command failed:', ' '.join(str(part) for part in command))
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()
    return result


def find_existing_repo():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src' / 'jspace_research').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    return None


WORKSPACE = Path('/content') if in_colab() else Path.home() / 'jspace-workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path('/content/jspace-research') if in_colab() else (find_existing_repo() or WORKSPACE / 'jspace-research')
BIPIA_CHECKOUT = WORKSPACE / 'BIPIA'
AGENTDOJO_CHECKOUT = WORKSPACE / 'agentdojo'
INJECAGENT_CHECKOUT = WORKSPACE / 'InjecAgent'
print('Using workspace:', WORKSPACE)
print('Using research repo:', REPO_ROOT)

if not REPO_ROOT.exists():
    run_checked(['git', 'clone', '--branch', RESEARCH_REVISION, RESEARCH_REPO, str(REPO_ROOT)])
elif (REPO_ROOT / '.git').is_dir():
    run_checked(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', RESEARCH_REVISION])
    run_checked(['git', '-C', str(REPO_ROOT), 'checkout', RESEARCH_REVISION])
    run_checked(['git', '-C', str(REPO_ROOT), 'merge', '--ff-only', f'origin/{RESEARCH_REVISION}'])
else:
    print('Existing local checkout without .git; installing from', REPO_ROOT)
if not BIPIA_CHECKOUT.exists():
    run_checked(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', str(BIPIA_CHECKOUT)])
run_checked(['git', '-C', str(BIPIA_CHECKOUT), 'checkout', BIPIA_REVISION])
if not AGENTDOJO_CHECKOUT.exists():
    run_checked(['git', 'clone', 'https://github.com/ethz-spylab/agentdojo.git', str(AGENTDOJO_CHECKOUT)])
run_checked(['git', '-C', str(AGENTDOJO_CHECKOUT), 'checkout', AGENTDOJO_REVISION])
if not INJECAGENT_CHECKOUT.exists():
    run_checked(['git', 'clone', 'https://github.com/uiuc-kang-lab/InjecAgent.git', str(INJECAGENT_CHECKOUT)])
run_checked(['git', '-C', str(INJECAGENT_CHECKOUT), 'checkout', INJECAGENT_REVISION])
run_checked(['pip', 'install', '-q', '-e', str(REPO_ROOT) + '[phase4]'])
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip() if (REPO_ROOT / '.git').is_dir() else 'local checkout (no git metadata)')
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('AgentDojo revision:', subprocess.check_output(['git', '-C', str(AGENTDOJO_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('InjecAgent revision:', subprocess.check_output(['git', '-C', str(INJECAGENT_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

fatal: could not create leading directories of '/content/jspace-research': Permission denied


CalledProcessError: Command '['git', 'clone', '--branch', 'prompt-injection-experiment', 'https://github.com/ethanncyb/jspace-research.git', '/content/jspace-research']' returned non-zero exit status 128.

## 2. Authenticate

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter judge credential loaded from Colab Secrets.')

## 3. Configure one persistent run directory

In [ ]:
RUN_MODE = 'full'  # keep 'smoke' only to re-run the short EmailQA pipeline
USE_DRIVE = True
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
RUN_NAME = f'jspace-{RUN_MODE}-qwen'

DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
DATA_ROOT_CANDIDATES = [
    DRIVE_MYDRIVE / 'jspace-research' / 'data',
    DRIVE_MYDRIVE / 'Colab Notebooks' / 'jspace-research' / 'data',
]

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = DRIVE_MYDRIVE / 'jspace-research' / 'runs' / RUN_NAME
else:
    RUN_ROOT = Path('/content') / RUN_NAME

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
PHASE3_DIR = RUN_ROOT / 'phase3'
PHASE4_DIR = RUN_ROOT / 'phase4'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / f'phase1_{RUN_MODE}_qwen.yaml'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Qwen config not found: {CONFIG_PATH}')


def _data_root_has_train(candidate):
    return (candidate / 'webqa' / 'train.jsonl').is_file() or (
        candidate / 'summarization' / 'train.jsonl'
    ).is_file()


def resolve_data_root():
    if not USE_DRIVE:
        return None
    for candidate in DATA_ROOT_CANDIDATES:
        if _data_root_has_train(candidate):
            return candidate
    for candidate in DATA_ROOT_CANDIDATES:
        if candidate.is_dir():
            return candidate
    return DATA_ROOT_CANDIDATES[0]


def resolve_named_file(name, filename):
    preferred = None if DATA_ROOT is None else DATA_ROOT / name / filename
    search_roots = []
    if DATA_ROOT is not None:
        search_roots.append(DATA_ROOT)
    search_roots.extend(candidate for candidate in DATA_ROOT_CANDIDATES if candidate != DATA_ROOT)
    for root in search_roots:
        path = root / name / filename
        if path.is_file():
            return path
    return preferred


def install_drive_test(source, destination):
    if source is None or not source.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f'Keeping existing {destination}')
        return
    destination.symlink_to(source)
    print(f'Linked {source} -> {destination}')


DATA_ROOT = resolve_data_root()
if RUN_MODE == 'full':
    WEBQA_TRAIN_PATH = resolve_named_file('webqa', 'train.jsonl')
    SUMMARIZATION_TRAIN_PATH = resolve_named_file('summarization', 'train.jsonl')
    install_drive_test(resolve_named_file('webqa', 'test.jsonl'), BIPIA_ROOT / 'qa' / 'test.jsonl')
    install_drive_test(
        resolve_named_file('summarization', 'test.jsonl'),
        BIPIA_ROOT / 'abstract' / 'test.jsonl',
    )
else:
    WEBQA_TRAIN_PATH = None
    SUMMARIZATION_TRAIN_PATH = None


def run_command(command, label):
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')

print('RUN_MODE:', RUN_MODE)
print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)
print('DATA_ROOT:', DATA_ROOT)
if WEBQA_TRAIN_PATH is not None:
    print('WebQA train:', WEBQA_TRAIN_PATH, 'exists=' + str(WEBQA_TRAIN_PATH.is_file()))
if SUMMARIZATION_TRAIN_PATH is not None:
    print('Summarization train:', SUMMARIZATION_TRAIN_PATH, 'exists=' + str(SUMMARIZATION_TRAIN_PATH.is_file()))

## 4. Confirm Drive and benchmark folders

This cell lists every path the full pipeline uses. A check means the file or directory is present. An `X` means it was not found. Missing Phase 1 inputs stop the notebook here so a full run does not start without WebQA, Summarization, and the BIPIA train files.

In [ ]:
import os
from pathlib import Path


def _mark(ok):
    return '[✓]' if ok else '[X]'


def _exists(path):
    return path is not None and path.exists()


def _check(label, path, required=False):
    ok = _exists(path)
    display_path = str(path) if path is not None else '(not found under DATA_ROOT)'
    if not ok and path is not None:
        display_path = f'{display_path}  (not found)'
    print(f'{_mark(ok)} {label}  {display_path}')
    return {'label': label, 'ok': ok, 'required': required, 'path': path}


print('Folders visible under DATA_ROOT:')
if DATA_ROOT is None:
    print('  (Drive data root was not resolved; USE_DRIVE is False or Drive is unmounted)')
elif not DATA_ROOT.is_dir():
    print(f'  {DATA_ROOT} does not exist')
else:
    children = sorted(DATA_ROOT.iterdir(), key=lambda item: item.name.lower())
    if not children:
        print(f'  {DATA_ROOT} is empty')
    for child in children:
        kind = 'dir' if child.is_dir() else 'file'
        print(f'  [{kind}] {child.name}')

print()
print('Drive / run')
rows = [
    _check('Drive mount', DRIVE_MYDRIVE, required=USE_DRIVE),
    _check('DATA_ROOT', DATA_ROOT, required=RUN_MODE == 'full'),
    _check('RUN_ROOT', RUN_ROOT),
    _check('Qwen config', CONFIG_PATH, required=True),
]

if USE_DRIVE:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    writable = os.access(RUN_ROOT, os.W_OK)
    print(f'{_mark(writable)} RUN_ROOT writable  {RUN_ROOT}')
    rows.append({'label': 'RUN_ROOT writable', 'ok': writable, 'required': True, 'path': RUN_ROOT})

print()
print('Phase 1 BIPIA train inputs')
rows.extend(
    [
        _check('BIPIA_ROOT', BIPIA_ROOT, required=True),
        _check('email/train.jsonl', BIPIA_ROOT / 'email' / 'train.jsonl', required=True),
        _check('table/train.jsonl', BIPIA_ROOT / 'table' / 'train.jsonl', required=True),
        _check('code/train.jsonl', BIPIA_ROOT / 'code' / 'train.jsonl', required=True),
        _check('text_attack_train.json', BIPIA_ROOT / 'text_attack_train.json', required=True),
        _check('code_attack_train.json', BIPIA_ROOT / 'code_attack_train.json', required=True),
        _check('WebQA train.jsonl', WEBQA_TRAIN_PATH, required=RUN_MODE == 'full'),
        _check('Summarization train.jsonl', SUMMARIZATION_TRAIN_PATH, required=RUN_MODE == 'full'),
    ]
)

print()
print('Phase 4 official tests and transfer checkouts')
phase4_rows = [
    _check('email/test.jsonl', BIPIA_ROOT / 'email' / 'test.jsonl'),
    _check('qa/test.jsonl', BIPIA_ROOT / 'qa' / 'test.jsonl'),
    _check('table/test.jsonl', BIPIA_ROOT / 'table' / 'test.jsonl'),
    _check('abstract/test.jsonl', BIPIA_ROOT / 'abstract' / 'test.jsonl'),
    _check('code/test.jsonl', BIPIA_ROOT / 'code' / 'test.jsonl'),
    _check('text_attack_test.json', BIPIA_ROOT / 'text_attack_test.json'),
    _check('code_attack_test.json', BIPIA_ROOT / 'code_attack_test.json'),
    _check('AgentDojo checkout', AGENTDOJO_CHECKOUT),
    _check('InjecAgent checkout', INJECAGENT_CHECKOUT),
    _check('InjecAgent/data', INJECAGENT_CHECKOUT / 'data'),
]
rows.extend(phase4_rows)

missing_required = [row['label'] for row in rows if row['required'] and not row['ok']]
missing_phase4 = [row['label'] for row in phase4_rows if not row['ok']]
print()
if missing_phase4:
    print('Phase 4-only items missing (warning, not fatal here):')
    for label in missing_phase4:
        print(f'  - {label}')
else:
    print('All Phase 4 checklist items were found.')

if missing_required:
    raise FileNotFoundError(
        f'{RUN_MODE} run is missing required Phase 1 inputs: ' + ', '.join(missing_required)
    )
print('Required Phase 1 inputs are present.')

## 5. Verify the GPU runtime

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before running the experiment.'
print('GPU:', torch.cuda.get_device_name(0))

## 6. Run or resume Phase 1

This freezes the manifest, captures activations, reconstructs J-space, and selects the layer. Rerunning the cell reuses compatible caches.

In [ ]:
phase1_command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--bipia-root', str(BIPIA_ROOT),
    '--output-dir', str(PHASE1_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    phase1_command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    phase1_command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])
run_command(phase1_command, 'Phase 1')

## 7. Inspect Phase 1 before continuing

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(Image(filename=str(PHASE1_DIR / 'layer_auprc.png')))
display(Image(filename=str(PHASE1_DIR / 'selected_layer_score_distribution.png')))

## 8. Run or resume Phase 2 generation

This GPU stage reads the frozen Phase 1 directory directly and runs three conditions: intact (`alpha=0.0`), partial removal (`alpha=0.5`), and full removal (`alpha=1.0`). The selected artifact path below is the only notebook-level connection between phases.

In [ ]:
phase2_base = [
    'jspace-phase2',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE2_DIR),
]
run_command([*phase2_base, '--stage', 'generate'], 'Phase 2 generation')

## 9. Run or resume Phase 2 analysis

This stage uses cached generations, ROUGE scoring, and the pinned API judge. It can also be run later on a CPU machine after copying the complete run directory.

In [ ]:
run_command([*phase2_base, '--stage', 'analyze'], 'Phase 2 analysis')

## 10. Inspect Phase 2 results

In [ ]:
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## 11. Construct and inspect the Phase 3 detectors

This CPU-only stage reads the frozen Phase 1 handoff directly. It does not load Qwen or the lens and does not depend on Phase 2.

In [ ]:
phase3_command = [
    'jspace-phase3',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE3_DIR),
]
run_command(phase3_command, 'Phase 3')
display(pd.read_csv(PHASE3_DIR / 'phase3_metrics.csv'))
display(Image(filename=str(PHASE3_DIR / 'phase3_detector_comparison.png')))

## 12. Run or resume and inspect Phase 4

This runs the three frozen transfer benchmarks. Generation requires CUDA; analysis uses CPU and OpenRouter only for BIPIA semantic outcomes.

In [ ]:
import pandas as pd
from IPython.display import Image, display

phase4_base = [
    'jspace-phase4',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--phase3', str(PHASE3_DIR),
    '--bipia-root', str(BIPIA_ROOT),
    '--agentdojo-root', str(AGENTDOJO_CHECKOUT),
    '--injecagent-root', str(INJECAGENT_CHECKOUT),
    '--output-dir', str(PHASE4_DIR),
]
run_command([*phase4_base, '--stage', 'generate'], 'Phase 4 generation')
run_command([*phase4_base, '--stage', 'analyze'], 'Phase 4 analysis')
display(pd.read_csv(PHASE4_DIR / 'phase4_metrics.csv'))
display(Image(filename=str(PHASE4_DIR / 'phase4_detector_transfer.png')))

## 13. Confirm persistence

With `USE_DRIVE = True`, all caches and results are already saved in Drive. With ephemeral `/content` storage, copy or download the entire run root before the Colab runtime ends. Preserve the `phase1/`, `phase2/`, `phase3/`, and `phase4/` directories together.

In [ ]:
import json

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print('Complete run root:', RUN_ROOT)
print('Phase 1 selected layer:', selection['selected_layer'])
print('Phase 2 results:', PHASE2_DIR / 'phase2_results.parquet')
print('Phase 3 metrics:', PHASE3_DIR / 'phase3_metrics.csv')
print('Phase 4 metrics:', PHASE4_DIR / 'phase4_metrics.csv')
assert (PHASE1_DIR / 'selected_layer.json').is_file()
assert (PHASE2_DIR / 'phase2_results.parquet').is_file()
assert (PHASE3_DIR / 'mean_detector.pt').is_file()
assert (PHASE3_DIR / 'logistic_detector.pt').is_file()
assert (PHASE4_DIR / 'phase4_predictions.parquet').is_file()

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. Phase 3 measures development-set detectability and freezes thresholds. Phase 4 evaluates those frozen detectors on held-out and transfer benchmarks without tuning. None of these phases establishes injection-specific causality.